# GlobalCLIP -- 02b: Train QLayer Interference Model

Trains `GlobalCLIPQLayerModel`: a frozen PARNET backbone + `MixCoeffHead`
+ `QLayer` (quantum interference) + dilated CNN refinement.

**Architecture:**
```
RNA sequence (4×600)
    │
    ▼
PARNET backbone  [frozen]
    │              │
    ▼              ▼
(B,512,L)      (B,223,L) RBP log-prob tracks
    │              │
    ▼              │ × exp(log_scale)
MixCoeffHead → alpha (B,223)   [sigmoid]
                   │
                   ▼
    QLayer  ψ_i = A_i · e^{iφ_i}      (223 learnable phases)
    I(p) = |Σ_i α_i · ψ_i(p)|²  →  (B,1,L)
                   │
          Dilated CNN refinement
                   │
                   ▼
            (B,1,L) GlobalCLIP prediction
```

The QLayer cross-terms `2·α_i·α_j·A_i·A_j·cos(φ_i−φ_j)` capture protein-protein
interactions with only 223 phase parameters.  After training on log-FE signal,
background-noise proteins naturally converge to φ ≈ φ_signal + π (destructive
interference → self-cancelling).

**Training target:** log(1+signal) − log(1+control)

**Loss:** Pearson + Multinomial NLL + alpha sparsity + phase L2 regularisation


## Set-up

### Imports

In [ ]:
import pylbsr.notebooks
import pylbsr.misc

import torch
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from pathlib import Path
from dotmap import DotMap

from parnet_additional_utils import (
    load_parnet_model,
    ParnetModelName,
)
from globalclip_utils import (
    GlobalCLIPQLayerModel,
    GlobalCLIPLightningModule,
    GlobalCLIPDataset,
    save_run_config,
)


### Initialisation

In [ ]:
_notebook_name = "02b_train_qlayer.py.ipynb"
_notebook_path = f"notebooks/globalclip/{_notebook_name}"

pylbsr.notebooks.enable_cell_timing_metadata(show=True)
logger = pylbsr.misc.init_logger(_notebook_name)
# Walks up from the notebook path until it finds the repo root, so all later
# paths are relative to the project regardless of the Jupyter working dir.
PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")

### Parameters

In [ ]:
params_gpu_index          = 0
params_run_id             = "globalclip.qlayer.v1"

# Dataset
params_control_dataset    = "globalclip_lysate_noNHS"   # cleanest background
params_seq_length         = 600
params_batch_size         = 64
params_num_rbps           = 223

# Model -- MixCoeffHead
params_mix_hidden         = 128

# Model -- CNN after QLayer
params_cnn_channels       = 64
params_cnn_kernel         = 9
params_cnn_layers         = 3    # uses dilation 1, 2, 4

# Training
params_lr                 = 1e-4
params_max_epochs         = 50
params_lambda_nll         = 0.3
params_lambda_alpha       = 5.0
params_lambda_phase       = 0.01   # L2 regularisation on QLayer phases
params_early_stop_patience= 8
params_num_workers        = 4


### Filepaths and device

In [ ]:
pylbsr.misc.set_seed(42)

device = torch.device(f"cuda:{params_gpu_index}" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(params_gpu_index)
    logger.info(f"GPU: {torch.cuda.get_device_name(device)}")
else:
    logger.warning("No GPU available, running on CPU (will be slow).")

_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.server.yaml").read_text())
pretrained_model_name = ParnetModelName.PARNET_7M_0_0

def _res(p):
    # filepaths.server.yaml stores paths relative to the project root.
    p = Path(p)
    return p if p.is_absolute() else PROJECT_DIR / p

FILEPATHS = DotMap()
FILEPATHS.pretrained_model = _res(_fp_cfg["models"][pretrained_model_name.value])
FILEPATHS.dataset          = _res(_fp_cfg["data"][params_control_dataset]["pt"])
FILEPATHS.rbp_names        = PROJECT_DIR / "results" / "globalclip" / "datasets" / "rbp_names.txt"
FILEPATHS.output_dir       = PROJECT_DIR / _fp_cfg["results"]["qlayer_model"] / params_run_id
FILEPATHS.output_dir.mkdir(parents=True, exist_ok=True)

for k, v in FILEPATHS.items():
    logger.info(f"{k:25s}: {v}")

## Load data

In [ ]:
train_ds = GlobalCLIPDataset(FILEPATHS.dataset, split="train",
                              seq_len=params_seq_length, total_key="globalCLIP")
val_ds   = GlobalCLIPDataset(FILEPATHS.dataset, split="valid",
                              seq_len=params_seq_length, total_key="globalCLIP")

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=params_batch_size, shuffle=True,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=params_batch_size, shuffle=False,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)

logger.info(f"Train: {len(train_ds)} samples  Val: {len(val_ds)} samples")

# Sanity check: peek at one batch's tensor shapes before starting real training.
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"  batch['{k}']: {tuple(v.shape)}")

## Build QLayer model

In [ ]:
logger.info(f"Loading pretrained PARNET from {FILEPATHS.pretrained_model}")
parnet = load_parnet_model(
    pretrained_model_name,
    FILEPATHS.pretrained_model,
    dtype=torch.float32,
    device=device,
)
# Frozen throughout: only the QLayer combination layer below is trained.
parnet.eval()
logger.info("PARNET loaded (223-task head kept frozen).")

In [ ]:
model = GlobalCLIPQLayerModel(
    parnet_model=parnet,
    num_rbps=params_num_rbps,
    mix_hidden=params_mix_hidden,
    cnn_channels=params_cnn_channels,
    cnn_kernel=params_cnn_kernel,
    cnn_layers=params_cnn_layers,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
logger.info(f"Parameters: {trainable:,} trainable / {total:,} total")
logger.info(f"  -- MixCoeffHead: {sum(p.numel() for p in model.mix_coeff.parameters()):,}")
logger.info(f"  -- QLayer phases: {model.qlayer.phase.numel()} (one per RBP)")
logger.info(f"  -- CNN: {sum(p.numel() for p in model.cnn.parameters()):,}")

# Sanity check
with torch.no_grad():
    pred, alpha = model(batch["sequence"].to(device))
print(f"pred shape : {tuple(pred.shape)}")
print(f"alpha shape: {tuple(alpha.shape)}")
print(f"Initial phases (first 5): {model.qlayer.phase[:5].detach().cpu().numpy()}")


## Train

In [ ]:
# Lightning wraps the raw GlobalCLIPQLayerModel with the training loop, the
# composite loss (Pearson + lambda_nll * NLL + lambda_alpha * sparsity +
# lambda_phase * phase L2 reg), optimizer, and logging.
lightning_model = GlobalCLIPLightningModule(
    model=model,
    lr=params_lr,
    lambda_nll=params_lambda_nll,
    lambda_alpha=params_lambda_alpha,
    lambda_phase=params_lambda_phase,
)

callbacks = [
    # Keeps the 2 checkpoints with lowest val loss, not just the last epoch.
    ModelCheckpoint(
        dirpath=FILEPATHS.output_dir / "checkpoints",
        filename="best-{epoch:02d}-{val/loss:.4f}",
        monitor="val/loss",
        mode="min",
        save_top_k=2,
    ),
    EarlyStopping(
        monitor="val/loss",
        patience=params_early_stop_patience,
        mode="min",
    ),
]

loggers = [
    CSVLogger(str(FILEPATHS.output_dir), name="csv_logs"),
    TensorBoardLogger(str(FILEPATHS.output_dir), name="tb_logs"),
]

trainer = pl.Trainer(
    max_epochs=params_max_epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=[params_gpu_index] if torch.cuda.is_available() else 1,
    callbacks=callbacks,
    logger=loggers,
    log_every_n_steps=50,
    deterministic=True,
)

logger.info("Starting training...")
trainer.fit(lightning_model, train_loader, val_loader)
logger.info("Training complete.")

## Save model and run config

In [ ]:
# state_dict is the canonical save, reload with
# GlobalCLIPQLayerModel(...).load_state_dict(...); model.full.pt (the whole
# pickled object) is kept only for quick ad-hoc prototyping.
torch.save(model.state_dict(), FILEPATHS.output_dir / "model.statedict.pt")
torch.save(model, FILEPATHS.output_dir / "model.full.pt")

# run_config.json is read back by evaluate_new_models.py / export_results_data.py
# to reconstruct this exact architecture (num_rbps, cnn_*, lambdas, etc.)
# without having to pass all the same flags again.
run_cfg = {
    "model_type":            "GlobalCLIPQLayerModel",
    "pretrained_model_name":  pretrained_model_name.value,
    "control_dataset":        params_control_dataset,
    "params_seq_length":      params_seq_length,
    "params_batch_size":      params_batch_size,
    "params_num_rbps":        params_num_rbps,
    "params_mix_hidden":      params_mix_hidden,
    "params_cnn_channels":    params_cnn_channels,
    "params_cnn_kernel":      params_cnn_kernel,
    "params_cnn_layers":      params_cnn_layers,
    "params_lr":              params_lr,
    "params_max_epochs":      params_max_epochs,
    "params_lambda_nll":      params_lambda_nll,
    "params_lambda_alpha":    params_lambda_alpha,
    "params_lambda_phase":    params_lambda_phase,
    "dataset_path":           str(FILEPATHS.dataset),
    "output_dir":             str(FILEPATHS.output_dir),
}
save_run_config(FILEPATHS.output_dir, run_cfg)
logger.info(f"Saved to {FILEPATHS.output_dir}")

## Quick phase inspection

In [ ]:
rbp_names = (FILEPATHS.rbp_names).read_text().strip().split("\n") if FILEPATHS.rbp_names.exists() else [f"RBP_{i}" for i in range(223)]

phases = model.qlayer.phase.detach().cpu().numpy()
coupling = model.get_coupling_matrix().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of phases
axes[0].hist(phases, bins=30, color="steelblue", edgecolor="none", alpha=0.8)
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Phase φ_i (radians)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of learned QLayer phases\n"
                  "φ≈0: constructive;  φ≈±π: destructive (noise suppression)")

# Polar plot of phases
ax_pol = fig.add_axes([0.55, 0.1, 0.4, 0.8], polar=True)
ax_pol.scatter(phases, np.ones_like(phases), alpha=0.5, s=20, color="steelblue")
ax_pol.set_rticks([])
ax_pol.set_title("Polar: protein phases", pad=15)

plt.savefig(FILEPATHS.output_dir / "phase_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Phase range: [{phases.min():.3f}, {phases.max():.3f}] rad")
print(f"Std dev of phases: {phases.std():.3f}")


## Training curves

In [ ]:
csv_log_dir = FILEPATHS.output_dir / "csv_logs"
# Lightning starts a new version_N/ folder each time training is restarted
# in this output dir, so always take the latest one.
metrics_paths = sorted(csv_log_dir.glob("version_*/metrics.csv"))
if metrics_paths:
    df_csv = pd.read_csv(metrics_paths[-1])
    # Keep only the last logged row per epoch, drops intra-epoch step noise.
    epoch_df = df_csv.groupby("epoch").last().reset_index()

    plots = [
        ({"train": "train/loss_epoch", "val": "val/loss"}, "Total loss"),
        ({"train": "train/pearson_epoch", "val": "val/pearson"}, "Pearson loss"),
        # phase_reg should trend toward 0 for phases the model does not need;
        # a flat, non-zero curve here would suggest the phase-init bug is back.
        ({"val phase_reg": "val/phase_reg"}, "Phase L2 regularisation"),
    ]
    fig, axes = plt.subplots(1, len(plots), figsize=(14, 4))
    for (col_dict, title), ax in zip(plots, axes):
        for label, col in col_dict.items():
            if col in epoch_df.columns:
                epoch_df.plot("epoch", col, ax=ax, label=label, marker="o", markersize=3)
        ax.set_title(title)
        ax.legend(fontsize=8)
    plt.suptitle(f"Training metrics -- {params_run_id}")
    plt.tight_layout()
    plt.savefig(FILEPATHS.output_dir / "training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()

## Next steps

- Run `03_evaluate_and_analyze.py.ipynb` to compare both models on the test set
- Inspect the coupling matrix `model.get_coupling_matrix()` to see which proteins cooperate
- The polar plot of phases reveals cooperative clusters (same φ) vs noise suppressors (φ+π)
